# Progetto iniziale: immagini sintetiche e CNN

Materiale originale **AI Lab**, non una lezione Kaggle. Codice in inglese; attività e spiegazioni in italiano.

Esegui le celle in ordine con **Maiusc + Invio**. Salva una copia prima di modificare il codice.


## Obiettivi
Classificare barre orizzontali e verticali in immagini sintetiche 16×16. Dati originali AI Lab generati qui, nessun download né GPU. Richiede NumPy e TensorFlow, esecuzione locale non verificata. Dopo aver capito la pipeline puoi sostituire i dati con immagini autorizzate.


In [ ]:
import os
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)
rng = np.random.default_rng(42)
images = rng.normal(0, 0.05, (300,16,16,1)).astype("float32")
labels = np.arange(300) % 2
for index,label in enumerate(labels):
    location = rng.integers(3,12)
    if label == 0:
        images[index,location:location+2,:,0] += 1
    else:
        images[index,:,location:location+2,0] += 1
order = rng.permutation(300)
images, labels = images[order], labels[order]
model = keras.Sequential([keras.Input(shape=(16,16,1)), layers.Conv2D(8,3,activation="relu"), layers.GlobalAveragePooling2D(), layers.Dense(1,activation="sigmoid")])
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["binary_accuracy"])
options = tf.data.Options()
options.threading.private_threadpool_size = 1
train_data = tf.data.Dataset.from_tensor_slices((images[:180], labels[:180])).batch(16).with_options(options)
valid_data = tf.data.Dataset.from_tensor_slices((images[180:240], labels[180:240])).batch(16).with_options(options)
model.fit(train_data,validation_data=valid_data,epochs=30,verbose=0)
predictions = model(images[240:],training=False).numpy().ravel() >= 0.5
print("Test accuracy:",np.mean(predictions == labels[240:]))


## Esercizi
Confronta la CNN con una baseline. Aggiungi rumore e variazioni di posizione; non usare rotazioni che scambiano le etichette. Se usi immagini reali, separa soggetti e sequenze fra gli split e registra le licenze.

## Interfaccia
Caricamento di immagine, conversione a 16×16 in scala di grigi e probabilità; possibile con Gradio in un ambiente Python esterno al sito.
